# 3.1.5 word2vec / Item2Vec：行为序列嵌入与召回

> 阅读版与 Web 应用内容一致；实验数值来自本 Notebook 的已执行输出。

## Goal

单独掌握用 word2vec 思想把行为序列学成物品向量：从 Skip-gram 与负采样、真实 MovieLens 序列训练，到向量召回与 Recall@K 评测。

## Setup

本 Notebook 的默认真实数据是 **GroupLens MovieLens latest-small：经典评分与邻域任务**。`smoke` 档读取仓库内可审计的确定性切片，`full` 档扩大到官方完整文件；两档都不制造交互、曝光、标签或行为序列。切片规则、源地址、哈希与许可记录在 `data/README.md` 及对应数据目录。

**主要资料：** [Efficient Estimation of Word Representations in Vector Space](https://arxiv.org/abs/1301.3781)

In [ ]:
from pathlib import Path
import os, sys, json
import torch
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
ARTIFACT_ROOT = Path(os.environ.get("RECSYS_ARTIFACT_ROOT", PROJECT_ROOT)).expanduser().resolve()
sys.path.insert(0, str(PROJECT_ROOT))
os.environ.setdefault("RECSYS_PROFILE", "full")
PROFILE = os.environ["RECSYS_PROFILE"]
CUDA_AVAILABLE = torch.cuda.is_available()
from recsys_lab.data import (load_movielens, movielens_provenance, load_amazon_2023,
                             amazon_provenance, load_kuairand, kuairand_provenance,
                             load_amazon_2018, amazon_2018_provenance,
                             load_movielens_1m, movielens_1m_provenance,
                             load_mind_amazon_books, mind_amazon_provenance,
                             load_census_income, census_income_provenance)
DATASET_KEY = "movielens"
if DATASET_KEY == "movielens":
    real_ratings, real_movies = load_movielens()
    real_interactions = real_ratings
    REAL_DATASET = movielens_provenance(real_ratings)
elif DATASET_KEY == "amazon-books":
    real_ratings = load_amazon_2018("Books") if PROFILE == "full" else load_amazon_2023()
    real_interactions, real_movies = real_ratings, None
    REAL_DATASET = amazon_2018_provenance(real_ratings) if PROFILE == "full" else amazon_provenance(real_ratings)
elif DATASET_KEY == "mind-amazon-books":
    real_ratings = load_mind_amazon_books() if PROFILE == "full" else load_amazon_2023()
    real_interactions, real_movies = real_ratings, None
    REAL_DATASET = mind_amazon_provenance(real_ratings) if PROFILE == "full" else amazon_provenance(real_ratings)
elif DATASET_KEY == "amazon-electronics":
    real_ratings = load_amazon_2018("Electronics") if PROFILE == "full" else load_amazon_2023()
    real_interactions, real_movies = real_ratings, None
    REAL_DATASET = amazon_2018_provenance(real_ratings) if PROFILE == "full" else amazon_provenance(real_ratings)
elif DATASET_KEY == "movielens-1m":
    real_ratings = load_movielens_1m() if PROFILE == "full" else load_amazon_2023()
    real_interactions, real_movies = real_ratings, None
    REAL_DATASET = movielens_1m_provenance(real_ratings) if PROFILE == "full" else amazon_provenance(real_ratings)
elif DATASET_KEY == "census-income" and PROFILE == "full":
    census_train_x, census_train_y, census_test_x, census_test_y = load_census_income()
    real_interactions, real_movies, real_ratings = census_train_x, None, census_train_x
    REAL_DATASET = census_income_provenance()
elif DATASET_KEY == "amazon-2023":
    real_ratings = load_amazon_2023()
    real_interactions, real_movies = real_ratings, None
    REAL_DATASET = amazon_provenance(real_ratings)
else:
    real_interactions, real_movies = load_kuairand()
    real_ratings = real_interactions
    REAL_DATASET = kuairand_provenance(real_interactions)
print({"profile": PROFILE, "project_root": str(PROJECT_ROOT), "artifact_root": str(ARTIFACT_ROOT), "real_dataset": REAL_DATASET,
       "cuda_available": CUDA_AVAILABLE,
       "cuda_device": torch.cuda.get_device_name(0) if CUDA_AVAILABLE else None})
assert REAL_DATASET["randomly_fabricated_rows"] == 0

## 来源论文与阅读提示

**Mikolov et al. (2013), Efficient Estimation of Word Representations in Vector Space** 提出 Skip-gram 与 CBOW：用一个词预测其上下文，在海量文本上高效学到词向量。推荐系统借这条思路形成 **Item2Vec**——把每个用户的正反馈物品序列当成一句话、物品当成词，于是同一序列里共现的物品会在嵌入空间被拉近，得到可做全库召回的物品向量。本实验用 MovieLens 正反馈序列训练 Skip-gram，再用向量近邻做下一物品 Recall@K。

### 公式怎么读（直觉版）

Skip-gram 的训练样本是 (中心, 上下文) 物品对。中心向量 $v_c$ 与上下文向量 $v_w$ 做内积，越共现越希望内积大；负采样再随机抽几个不共现的物品，希望它们的内积小。用 Sigmoid 把内积压到 0～1，配 LogLoss，就是“拉正、推负”。向量、点积与 Sigmoid 见 **3.0 推荐算法数学基础**。

## Steps

## 1. 数据集与序列构造

### 1.1 从真实评分到行为序列

用 MovieLens latest-small 的真实评分，按时间排序、保留 `rating >= 4` 的正反馈，得到每个用户的物品序列。`smoke` 档读取仓库内确定性切片，`full` 档扩大到官方完整文件；两档都不制造交互或序列。

In [ ]:
import numpy as np
import pandas as pd
import torch

SEED = 2026
torch.manual_seed(SEED)
from recsys_lab.data import load_movielens, leave_last_out, movielens_provenance

ratings, movies = load_movielens(max_users=80, max_items=360, min_user_events=12)
train_ratings, test_ratings = leave_last_out(ratings)
print({"rows": len(ratings), "users": ratings.user_id.nunique(), "items": ratings.item_id.nunique(),
       "source": movielens_provenance(ratings)["source"], "fabricated_rows": 0})
ratings[["userId", "movieId", "rating", "timestamp", "title"]].head(6)

### 1.2 为什么把序列当一句话？

协同过滤只看“谁和谁共现”，丢了次序；而用户的观看/购买往往有顺序迁移（看完 A 更可能看 B）。把序列当句子，Skip-gram 用窗口内的邻居作上下文，既利用共现也保留局部次序，学到的向量还能离线预计算、线上做 ANN 召回。

## 2. word2vec：Skip-gram 与负采样

### 2.1 直觉

对序列 `[i0, i1, i2, i3]`、窗口 1，生成正对 `(i0,i1),(i1,i0),(i1,i2),(i2,i1),(i2,i3),(i3,i2)`。每对训练“中心预测上下文”；再为每对配几个随机负物品。训练后取中心嵌入作为物品向量。

### 2.2 数学：负采样目标

**通用先修：** [3.0.1 隐式反馈与假负例](/notebooks/3_0_1_data_ml_basics#implicit-feedback) · [3.0.2 点积与 embedding](/notebooks/3_0_2_linear_algebra#matmul-embedding) · [3.0.5 二元交叉熵](/notebooks/3_0_5_information_theory#cross-entropy-kl)<br>
**本论文新增数学：** Skip-gram 的中心/上下文双 embedding、负采样目标，以及按出现次数的 $0.75$ 次幂构造噪声分布。

先说符号：每个物品有两套向量——作中心词时用 $v_c$，作上下文时用 $v_w$（分开学习更稳定，推理时只保留中心向量）；$\sigma(a)=1/(1+e^{-a})$ 把内积压成“会共现”的概率。对正对 $(c,w)$ 和负样本集合 $\mathcal{N}$：

$$
\mathcal{L}=-\log\sigma(v_c\cdot v_w)-\sum_{n\in\mathcal{N}}\log\sigma(-v_c\cdot v_n)
$$

第一项希望正对内积大（$\sigma(v_c\cdot v_w)$ 接近 1）；第二项希望负样本内积小（$\sigma(-v_c\cdot v_n)$ 接近 1，等价于 $\sigma(v_c\cdot v_n)$ 接近 0）。为什么不直接对整个物品表做 softmax？因为分母要遍历全部物品，代价随目录线性增长；负采样把它近似为几个二分类，训练代价与物品总数无关。

**手算一遍。** 取 $v_c=[1,0]$，正样本 $v_w=[0.8,0.2]$：内积 $=0.8$，$\sigma(0.8)\approx0.69$，$-\log0.69\approx0.37$。负样本 $v_n=[-1,1]$：内积 $=-1$，$\sigma(1)\approx0.73$，$-\log0.73\approx0.31$。合计 $\mathcal L\approx0.68$；训练会把 $v_c$ 推向 $v_w$、推离 $v_n$，两项随之变小。

内积越大、Sigmoid 越接近 1；负样本希望内积小、Sigmoid 接近 0。它等价于对正负样本做带 Logits 的二元交叉熵。

负物品也不是随便均匀抽。word2vec 论文实现常用 unigram 分布的 $0.75$ 次幂：若物品 $i$ 在训练上下文中出现 $c_i$ 次，

$$
P_n(i)=\frac{c_i^{0.75}}{\sum_j c_j^{0.75}}.
$$

$0.75<1$ 会压平头部：热门物品仍比冷门物品更常成为负例，但不会完全支配训练。被采为负例只表示“本次窗口没有把它当正上下文”，不证明用户讨厌它，因此仍可能是假负例。

还要标清迁移边界：Mikolov 论文研究的是文本词与语言上下文；**Item2Vec** 才把词换成物品、句子换成会话/行为篮子。可迁移的是“局部共现可学习 embedding”的结构，不能把词语的句法/语义结论直接宣称为用户偏好，也不能把未共现解释成真实负反馈。

### 2.3 小序列演示：玩具序列怎样学到共现

先用 3 个用户的小序列手算 Skip-gram 对，确认“同一序列共现的物品对”如何进入训练集。

In [ ]:
toy_sequences = [[1, 2, 3, 2], [2, 3, 4], [1, 3, 4, 5]]

def skip_gram_pairs(sequences, window=1):
    pairs = []
    for seq in sequences:
        for i, c in enumerate(seq):
            for j in range(max(0, i - window), min(len(seq), i + window + 1)):
                if j != i:
                    pairs.append((c, seq[j]))
    return pairs

toy_pairs = skip_gram_pairs(toy_sequences, window=1)
print("toy 正对数:", len(toy_pairs), "示例:", toy_pairs[:6])
assert (2, 3) in toy_pairs and (3, 2) in toy_pairs

### 2.4 训练：在真实 MovieLens 序列上训练 Skip-gram

In [ ]:
class SkipGram(torch.nn.Module):
    def __init__(self, num_items, dim=32):
        super().__init__()
        # 中心嵌入与上下文嵌入分离，是 word2vec 的常规做法；负采样时只更新相关行。
        self.center = torch.nn.Embedding(num_items, dim)
        self.context = torch.nn.Embedding(num_items, dim)
        torch.nn.init.uniform_(self.center.weight, -0.01, 0.01)
        torch.nn.init.zeros_(self.context.weight)

    def forward(self, c, w):
        # 中心向量与上下文向量的内积越大，说明两物品越可能在同一序列共现。
        return (self.center(c) * self.context(w)).sum(-1)


# 物品 ID +1（0 留作 padding）；按时间构造每个用户的训练序列。
positive = train_ratings[train_ratings.rating >= 4.0].sort_values(["user_id", "timestamp", "item_id"])
sequences = positive.groupby("user_id").item_id.apply(lambda v: [int(x) + 1 for x in v]).to_dict()
sequences = {int(u): s for u, s in sequences.items() if len(s) >= 4}
pairs = skip_gram_pairs(list(sequences.values()), window=3)
num_items = int(ratings.item_id.max()) + 2
center = torch.tensor([p[0] for p in pairs], dtype=torch.long)
context = torch.tensor([p[1] for p in pairs], dtype=torch.long)
rng = np.random.default_rng(SEED)

# 论文式噪声分布：上下文出现次数的 0.75 次幂，而不是均匀采样。
context_counts = np.bincount(context.numpy(), minlength=num_items)[1:].astype(float)
negative_sampling_probability = context_counts ** 0.75
negative_sampling_probability /= negative_sampling_probability.sum()

model = SkipGram(num_items, dim=32)
optimizer = torch.optim.Adam(model.parameters(), lr=0.05)
losses = []
NUM_NEG = 5
for _ in range(8):
    neg_items = torch.tensor(
        rng.choice(np.arange(1, num_items), size=(len(pairs), NUM_NEG), p=negative_sampling_probability),
        dtype=torch.long,
    )
    score_pos = model(center, context)
    score_neg = model(center[:, None].expand(-1, NUM_NEG).reshape(-1), neg_items.reshape(-1))
    pos = torch.nn.functional.binary_cross_entropy_with_logits(score_pos, torch.ones_like(score_pos))
    neg = torch.nn.functional.binary_cross_entropy_with_logits(score_neg, torch.zeros_like(score_neg))
    loss = pos + neg
    optimizer.zero_grad(); loss.backward(); optimizer.step()
    losses.append(float(loss.detach()))
print({"pairs": len(pairs), "num_items": num_items, "loss_first": round(losses[0], 4), "loss_last": round(losses[-1], 4)})

### 2.5 推理与召回：物品向量 -> 用户向量 -> Top-K

取中心嵌入作为物品向量，归一化后用余弦相似度。用户向量 = 其训练历史物品向量的均值；屏蔽已见物品后取 Top-K，检查留出的下一物品是否命中。

In [ ]:
with torch.no_grad():
    embeddings = model.center.weight.detach().numpy()
norms = np.linalg.norm(embeddings, axis=1, keepdims=True)
normed = embeddings / np.maximum(norms, 1e-12)

test_frame = test_ratings.sort_values("user_id")
targets = test_frame.item_id.to_numpy() + 1
users = test_frame.user_id.to_numpy()
seen = {u: set(s) for u, s in sequences.items()}
hits, all_recs = 0, []
with np.errstate(divide="ignore", invalid="ignore", over="ignore"):
    for user, target in zip(users, targets):
        history = sequences.get(int(user), [])
        if not history:
            continue
        uv = normed[history].mean(axis=0)
        uv = uv / max(float(np.linalg.norm(uv)), 1e-12)
        scores = normed @ uv
        scores[list(seen.get(int(user), []))] = -1.0
        k = min(5, len(scores) - 1)
        top5 = np.argpartition(-scores, kth=k)[:5]
        all_recs.extend(top5.tolist())
        hits += int(target in top5)
word2vec_recall_at_5 = hits / max(len(targets), 1)
word2vec_coverage = len(set(all_recs)) / num_items
print({"word2vec_recall@5": round(word2vec_recall_at_5, 4), "coverage": round(word2vec_coverage, 4), "num_test_users": len(targets)})

# 相似物品示例：任取一个热门物品，看它的向量最近邻。
mode_item = int(ratings.item_id.mode().iloc[0])
sims = normed @ normed[mode_item + 1]
sims[mode_item + 1] = -1.0
neighbors = np.argsort(-sims)[:5]
movie_lookup = ratings.drop_duplicates("item_id").set_index("item_id")["title"]
print("anchor:", movie_lookup.get(mode_item))
print("近邻:", [movie_lookup.get(i - 1) for i in neighbors])

### 2.6 结果讨论与边界

Recall@5 在小切片上可能不高：序列短、物品多、负采样随机性大。它仍是合理的召回基线，且物品向量可离线预计算、线上做 ANN。与 ItemCF 相比，向量召回更稠密、可泛化到未直接共现的物品；但冷启动物品无向量，窗口与负样本数对效果影响明显。

**优点**：向量可预计算、ANN 全库召回；序列保留局部次序；易扩展到多模态侧信息。
**缺点**：新物品无向量；短序列学不稳；共现继承曝光与热门偏置；窗口/负采样需调参。

**推理复杂度**：训练只更新被采样到的 embedding 行；线上物品向量离线预计算，用户向量由少量历史向量均值得到，全库检索交给 ANN。

## Checks

确认序列非空、loss 下降、Recall 范围合法，并比较 word2vec 与 ItemCF 的召回。

In [ ]:
assert len(pairs) > 0
assert losses[-1] < losses[0]
assert 0 <= word2vec_recall_at_5 <= 1
assert 0 <= word2vec_coverage <= 1
print('PASS：word2vec / Item2Vec 序列构造、Skip-gram 训练与向量召回均有效。')

In [ ]:
result_dir = ARTIFACT_ROOT / "results" / "chapter_3_1"; result_dir.mkdir(parents=True, exist_ok=True)
payload = {"records": [
    {"algorithm": "word2vec / Item2Vec", "task": "Top-K", "metric": "Recall@5 ↑", "value": word2vec_recall_at_5,
     "secondary_metric": "Coverage", "secondary_value": word2vec_coverage,
     "online_inference": "物品向量 ANN 召回", "source_notebook": "3_1_5_word2vec"},
]}
(result_dir / "word2vec.json").write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding="utf-8")
print("已写出章节指标：word2vec.json")

## Next Steps

扩大到 Amazon 完整序列、调窗口/负采样/维度，加入时间衰减与热门度惩罚，并与 MF 向量、SASRec 序列召回对比 Recall 与 Coverage。